# Supervised Learning: Memprediksi Resep/Notes Parfum (Random Forest)

Di notebook ini, kita menggunakan memuat dataset yang sudah diberi label klaster (dari proses **Unsupervised Learning (K-Means)** sebelumnya). Kita akan menggunakan algoritma **Supervised Learning (Random Forest)** untuk memprediksi klaster parfum baru berdasarkan wangi (deskripsi) atau karakteristiknya.

Dengan kata lain, kita mencoba memprediksi kombinasi resep (notes) dari deskripsi wangi!

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

import warnings
warnings.filterwarnings('ignore')

## 2. Load Dataset Berlabel Klaster

In [ ]:
# Load dataset yang sudah punya kolom 'Cluster_Aroma' dari proses K-Means sebelumnya
df = pd.read_csv('final_perfume_data_with_clusters.csv')

# Mengisi nilai kosong jika ada
df['Notes'] = df['Notes'].fillna('Unknown')
df['Description'] = df['Description'].fillna('Unknown')

print("Tabel dengan label Cluster (Supervised Target):")
display(df[['Name', 'Description', 'Notes', 'Cluster_Aroma']].head())

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x85 in position 24088: invalid start byte

## 3. Setup X (Features) dan Y (Target)

Sekarang `Cluster_Aroma` menjadi target (label) yang diawasi (Supervised). 
Fitur (X) diambil dari kata-kata (TF-IDF) berdasarkan *Description* saja, sehingga ketika kita punya wangi baru (Deskripsi), kita bisa tebak *Notes* nya.

In [ ]:
# Variabel Independen (X) - menggunakan Description saja
tfidf_supervised = TfidfVectorizer(stop_words='english', max_features=1000)
X = tfidf_supervised.fit_transform(df['Description'])

# Variabel Dependen / Target (y)
y = df['Cluster_Aroma'].values

# Membagi data menjadi Data Latih (Train) dan Data Uji (Test) - 80% Train, 20% Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Jumlah Data Latih:", X_train.shape[0])
print("Jumlah Data Uji:", X_test.shape[0])

## 4. Modeling (Random Forest Classifier)

In [ ]:
# Inisialisasi Classifier Murni (Supervised)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Melatih model dengan pasangan X_train dan y_train
rf_model.fit(X_train, y_train)

# Evaluasi model
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Akurasi Model Supervised: {accuracy * 100:.2f}%")
print("\nLaporan Klasifikasi:\n", classification_report(y_test, y_pred))

## 5. Simulasi Prediksi Resep dari Wangi Baru

Bagaimana jika kita mendapatkan ringkasan wangi parfum baru yang belum ada resepnya? Kita berikan teks `Description`-nya ke dalam model prediksi!

In [ ]:
# Coba wangi baru / deskripsi baru
wangi_baru = ["A bold mix of smoked leather and roasted coffee beans, warmed by sunset amber and vanilla."]

# Ubah menjadi TF-IDF (gunakan objek text-prep Supervised yang sudah di-fit sebelumnya)
wangi_baru_tfidf = tfidf_supervised.transform(wangi_baru)

# Prediksi klaster
prediksi_klaster = rf_model.predict(wangi_baru_tfidf)[0]
print(f"Teks wangi ini diprediksi masuk ke dalam: Klaster {prediksi_klaster}")

# Kita intip parfum apa saja dan mayoritas resep (Notes) yang ada di Klaster tersebut berdasar data awal
klaster_samples = df[df['Cluster_Aroma'] == prediksi_klaster].head(3)
print("\n--- Referensi Resep Parfum di Klaster Serupa ---")
for idx, row in klaster_samples.iterrows():
    print(f"- {row['Name']} (Brand: {row['Brand']})")
    print(f"  Bahan / Resep (Notes): {row['Notes']}\n")